# FIT5221 Assignment 1

- Name: Yin Zhang
- ID: [Student ID removed]


## Task 1: Harris corner detection (8 marks)

In [ ]:
import os
os.makedirs("output", exist_ok=True)

In [ ]:
# Complete this function
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy.signal import convolve2d
from skimage import color, filters, util
from skimage.feature import corner_peaks, corner_harris

def my_harris_corner_detector(img, block_size=2, ksize=3, k=0.04, min_distance=5):
    """
    Implement Harris corner detection
    Parameters:
        Gray: grayscale image (numpy array)
        Blocksize: Window size (not directly used in this implementation, but kept as a parameter)
        Ksize: Sobel operator size (fixed at 3, ignored)
        k: Harris response function parameters
        min_distance: NMS min distance
    return:
        kept: Corner response diagram after NMS
    """
    if len(img.shape) == 3:
        gray = color.rgb2gray(img)
    else:
        gray = img.copy()
    gray = util.img_as_float(gray)  
    # 1. Calculate gradient (Sobel operator)
    sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
    sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32)
    Ix = convolve2d(gray.astype(np.float32), sobel_x, mode='same')
    Iy = convolve2d(gray.astype(np.float32), sobel_y, mode='same')

    # 2. Calculate the second-order moment component
    Ix2 = Ix * Ix
    Iy2 = Iy * Iy
    Ixy = Ix * Iy

    # 3. Gaussian smoothing
    sigma = 1.0  # Adjustable smoothness level
    Ix2 = gaussian_filter(Ix2, sigma=sigma)
    Iy2 = gaussian_filter(Iy2, sigma=sigma)
    Ixy = gaussian_filter(Ixy, sigma=sigma)

    # 4. Calculate corner response R = det(M) - k * trace(M)^2
    det = Ix2 * Iy2 - Ixy * Ixy
    trace = Ix2 + Iy2
    R = det - k * trace * trace
    
    thresh = 0.01 * R.max()
    #keypoints_raw = np.argwhere(R > thresh)
    #keypoints = []
    #for y,x in keypoints_raw:
        #keypoints.append((x,y))
    
    #Non Maximum Suppression (NMS)
    #Sort in descending order by response value
    candidate_mask = R > thresh
    ys, xs = np.where(candidate_mask)
    responses = R[ys, xs]
    order = np.argsort(responses)[::-1]
    kept = []
    suppressed = np.zeros_like(candidate_mask, dtype=bool)
    min_distance=5
    for idx in order:
        y, x = ys[idx], xs[idx]
        if suppressed[y, x]:
            continue
        kept.append((x, y))
        y_min = max(0, y - min_distance)
        y_max = min(R.shape[0], y + min_distance + 1)
        x_min = max(0, x - min_distance)
        x_max = min(R.shape[1], x + min_distance + 1)
        suppressed[y_min:y_max, x_min:x_max] = True
    
    return kept

In [ ]:
def count_overlap(corners1, corners2, distance_thresh=2.5):
    """
    Calculate the number of overlaps between two sets of corner points 
    (Euclidean distance ≤ threshold considered as overlap).
    Corner1, Corner2: List of corner points, each element is (x, y)
    """
    if not corners1 or not corners2:
        return 0
    # Convert the list to an array for vectorization calculation
    arr1 = np.array(corners1)
    arr2 = np.array(corners2)
    # Calculate the Euclidean distance matrix between each corner point
    diff = arr1[:, np.newaxis, :] - arr2[np.newaxis, :, :]  # N1 x N2 x 2
    dist = np.linalg.norm(diff, axis=2)                     # N1 x N2
    overlap_count = 0
    for i in range(len(corners1)):
        if np.any(dist[i] <= distance_thresh):
            overlap_count += 1
    return overlap_count

In [ ]:
# Test the `my_harris_corner_detector` function with 6 images:
# - skimage.data.astronaut()
# - skimage.data.checkerboard()
# - castle1.jpg
# - castle2.jpg
# - monash1.jpg
# - monash2.jpg
# - singapore1.jpg
# - singapore2.jpg

from skimage import io, data, color, util
from skimage.feature import corner_harris, corner_peaks
import matplotlib.pyplot as plt
import os

os.makedirs("output", exist_ok=True)

images = [
    ('astronaut', data.astronaut()),
    ('checkerboard', data.checkerboard())
]

provided_images = [
    'castle1.jpg',
    'castle2.jpg',
    'monash1.jpg',
    'monash2.jpg',
    'singapore1.jpg',
    'singapore2.jpg'
]

for i in range(0, 6):
    img = io.imread('images/' + provided_images[i])
    images.append((provided_images[i], img))

k = 0.04
sigma = 1.0
thresh_ratio = 0.01

for img_name, img in images:
    print(f"\nprocess image: {img_name}")

    my_corners = my_harris_corner_detector(img)

    if len(img.shape) == 3:
        gray = color.rgb2gray(img)
    else:
        gray = img
    gray = util.img_as_float(gray)

    harris_response = corner_harris(gray, k=0.05, sigma=1.0)
    sk_corners_raw = corner_peaks(
        harris_response,
        min_distance=5,
        threshold_rel=0.01
    )
    sk_corners = [(c, r) for r, c in sk_corners_raw]

    # ----- Calculate the number of overlapping corner points -----
    overlap = count_overlap(my_corners, sk_corners, distance_thresh=2)

    # ----- visualization -----
    plt.figure(figsize=(10, 8))
    plt.imshow(img, cmap='gray' if img.ndim == 2 else None)

    # Draw custom corner points (red '+')
    for (x, y) in my_corners:
        plt.plot(x, y, 'r+', markersize=8, markeredgewidth=1.5)

    # Draw skimage corner points (blue 'o')
    for (x, y) in sk_corners:
        plt.plot(x, y, 'bo', markersize=6, fillstyle='none', markeredgewidth=1.5)

    plt.title(
        f"{img_name}\n"
        f"My corners: {len(my_corners)} | skimage corners: {len(sk_corners)} | Overlap: {overlap}"
    )
    plt.axis('off')
    plt.tight_layout()

    safe_name = img_name.replace(".jpg", "").replace(" ", "_")
    plt.savefig(f"output/task1_{safe_name}.png", dpi=300, bbox_inches='tight')
    plt.show()

    # Print statistical information
    print(f"  Customize the number of detected corners: {len(my_corners)}")
    print(f"  Number of skimage detected corner points: {len(sk_corners)}")
    print(f"  Number of overlapping corner points: {overlap}")

## Task 2. Homography estimation (2 marks)

In [ ]:
# Complete this function
def my_homography_estimation(src_pts, dst_pts):
    """
    Calculate the homography matrix H, satisfying dst=H * src (homogeneous coordinates)
    Src_pts, dst_pts: N x 2 array
    Return: 3x3 H-matrix
    """
    N = src_pts.shape[0]
    A = []
    for i in range(N):
        x, y = src_pts[i]
        u, v = dst_pts[i]
        A.append([x, y, 1, 0, 0, 0, -u*x, -u*y, -u])
        A.append([0, 0, 0, x, y, 1, -v*x, -v*y, -v])
    A = np.array(A, dtype=np.float64)
    # Solving the least squares solution for Ah=0
    _, _, Vt = np.linalg.svd(A)
    h = Vt[-1] / Vt[-1, -1]   # Normalize to make the last element 1
    H = h.reshape(3, 3)
    return H

## Task 3. RANSAC (6 marks)

In [ ]:
# Complete this function
def my_ransac(src_pts, dst_pts, threshold=4.0, max_iterations=2000):
    """
    RANSAC estimates homography matrix
    Src_pts, dst_pts: N x 1 x 2 array (OpenCV format)
    Threshold: interior point error threshold (pixels)
    Max_iterations: maximum number of iterations
    Return: (H, mask) where H is a 3x3 matrix and mask is a Boolean array of Nx1
    """
    src = src_pts.reshape(-1, 2)
    dst = dst_pts.reshape(-1, 2)
    n_points = src.shape[0]

    best_inliers = []
    best_H = None
    count = 1
    for _ in range(max_iterations):
        # Randomly select 4 pairs of points
        indices = np.random.choice(n_points, 4, replace=False)
        src_sample = src[indices]
        dst_sample = dst[indices]

        # Calculate the homography matrix
        H = my_homography_estimation(src_sample, dst_sample)
        if H is None:
            continue

        # Calculate the projection error of all points
        inliers = []
        for i in range(n_points):
            p = np.array([src[i, 0], src[i, 1], 1.0])
            proj = H @ p
            proj = proj / proj[2]
            error = np.linalg.norm(proj[:2] - dst[i])
            if error < threshold:
                inliers.append(i)
        inlier_count = len(inliers)

        print(f"Iter {count}: inliers = {inlier_count}, outliers = {n_points - inlier_count}")
        count=count+1
        if len(inliers) > len(best_inliers):
            best_inliers = inliers
            best_H = H
            # If enough inliers have been found, end early
            if len(best_inliers) > 0.9 * n_points:
                break

    # estimate using all interior points H
    if len(best_inliers) >= 4:
        src_inliers = src[best_inliers]
        dst_inliers = dst[best_inliers]
        H_final = my_homography_estimation(src_inliers, dst_inliers)
        #Generate Mask
        mask = np.zeros((n_points, 1), dtype=np.uint8)
        mask[best_inliers] = 1
        print(len(best_inliers))
        return H_final, mask
    else:
        return None, None

In [ ]:
# Test `my_ransac`  with P1, P2 as below
import numpy as np
P1 = np.asarray([[428, 171],
                  [547, 137],
                  [60,  90],
                  [397, 208],
                  [521, 224],
                  [563, 333],
                  [507, 178],
                  [476, 277],
                  [180, 353],
                  [543, 307],
                  [531, 98]])
P2 = np.asarray([[51, 152],
                [162, 98],
                [162,  98],
                [27, 194],
                [152, 189],
                [51, 152],
                [130, 145],
                [117, 248],
                [26, 434],
                [188, 266],
                [140, 63]])

H ,mask= my_ransac(P1, P2)
print("Transformation matrix:  ")
print(H) 

## Task 4. Image stitching (4 marks)


Test `my_image_stitching` function with three pair of input images:
- First test input:
  - image 1 is `castle1.jpg`
  - image 2 is `castle2.jpg`

  - Generate a stitched image from `image1` and `image2`, save this image under `stitched_castle.jpg`

- Second test input:
  - image 1 is `singapore1.jpg`
  - image 2 is `singapore2.jpg`.
  - Generate a stitched image from `image1` and `image2`, save this image under `stitched_singapore.jpg`

- Third test input:
  - image 1 is `monash1.jpg`
  - image 2 is `monash2.jpg`.
  - Generate a stitched image from `image1` and `image2`, save this image under `stitched_monash.jpg`

To write an image as a file, you can use function `skimage.io.imsave(image_name, stitched_image)`

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

os.makedirs("output", exist_ok=True)

# -------------------- RANSAC --------------------
def my_ransac(src_pts, dst_pts, threshold=4.0, max_iterations=1000):
    """
    RANSAC estimates homography matrix
    Src_pts, dst_pts: N x 1 x 2 array (OpenCV format)
    Return: (H, mask)
    """
    src = src_pts.reshape(-1, 2)
    dst = dst_pts.reshape(-1, 2)
    n_points = src.shape[0]

    if n_points < 4:
        return None, None

    best_inliers = []
    best_H = None

    for _ in range(max_iterations):
        indices = np.random.choice(n_points, 4, replace=False)
        src_sample = src[indices]
        dst_sample = dst[indices]

        H = my_homography_estimation(src_sample, dst_sample)
        if H is None:
            continue

        inliers = []
        for i in range(n_points):
            p = np.array([src[i, 0], src[i, 1], 1.0], dtype=np.float64)
            proj = H @ p

            if abs(proj[2]) < 1e-8:
                continue

            proj = proj / proj[2]
            error = np.linalg.norm(proj[:2] - dst[i])

            if error < threshold:
                inliers.append(i)

        if len(inliers) > len(best_inliers):
            best_inliers = inliers
            best_H = H

    if best_H is None or len(best_inliers) < 4:
        return None, None

    src_inliers = src[best_inliers]
    dst_inliers = dst[best_inliers]
    H_final = my_homography_estimation(src_inliers, dst_inliers)

    if H_final is None:
        return None, None

    mask = np.zeros((n_points, 1), dtype=np.uint8)
    mask[best_inliers] = 1

    return H_final, mask


# -------------------- Image Stitching --------------------
def my_image_stitching(img1, img2, pair_name, ratio=0.75, reproj_thresh=4.0):
    """
    Image stitching main function
    """
    # Convert to grayscale image
    if len(img1.shape) == 3:
        gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    else:
        gray1 = img1.copy()

    if len(img2.shape) == 3:
        gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
    else:
        gray2 = img2.copy()

    # ---------- Step 1: Harris corner detection ----------
    harris1 = my_harris_corner_detector(gray1, block_size=2, ksize=3, k=0.04)
    harris2 = my_harris_corner_detector(gray2, block_size=2, ksize=3, k=0.04)

    # Visualize Harris corner points
    img1_harris = img1.copy()
    img2_harris = img2.copy()

    for p in harris1:
        cv2.circle(img1_harris, p, 5, (0, 0, 255), -1)
    for p in harris2:
        cv2.circle(img2_harris, p, 5, (0, 0, 255), -1)

    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(img1_harris, cv2.COLOR_BGR2RGB))
    plt.title('Image 1 - Harris Corners')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(cv2.cvtColor(img2_harris, cv2.COLOR_BGR2RGB))
    plt.title('Image 2 - Harris Corners')
    plt.axis('off')

    plt.savefig(f"output/task4_harris_{pair_name}.png", dpi=300, bbox_inches='tight')
    plt.show()

    # ---------- Step 2: Extract SIFT descriptor ----------
    kp1 = [cv2.KeyPoint(float(p[0]), float(p[1]), 5) for p in harris1]
    kp2 = [cv2.KeyPoint(float(p[0]), float(p[1]), 5) for p in harris2]

    sift = cv2.SIFT_create()
    kp1, des1 = sift.compute(gray1, kp1)
    kp2, des2 = sift.compute(gray2, kp2)

    if des1 is None or des2 is None or len(kp1) < 4 or len(kp2) < 4:
        print("Insufficient number of Harris corner points to extract descriptors")
        return None

    # ---------- Step 3: feature matching ----------
    # BFMatcher + knnMatch + ratio test
    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
    matches = bf.knnMatch(des1, des2, k=2)

    good_matches = []
    for pair in matches:
        if len(pair) < 2:
            continue
        m, n = pair
        if m.distance < ratio * n.distance:
            good_matches.append(m)

    if len(good_matches) < 4:
        print("Insufficient matching points, unable to calculate homography matrix")
        return None

    # Visualize matching points
    matched_img = cv2.drawMatches(
        img1, kp1, img2, kp2, good_matches, None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )

    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(matched_img, cv2.COLOR_BGR2RGB))
    plt.title('Feature Matches (Harris + SIFT)')
    plt.axis('off')
    plt.savefig(f"output/task4_matches_{pair_name}.png", dpi=300, bbox_inches='tight')
    plt.show()

    # ---------- Step 4: RANSAC estimates homography matrix ----------
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)


    H, mask = my_ransac(dst_pts, src_pts, threshold=reproj_thresh)

    if H is None or mask is None:
        print("RANSAC failed to calculate homography matrix")
        return None

    inliers = mask.ravel().tolist()
    inlier_matches = [good_matches[i] for i in range(len(good_matches)) if inliers[i]]

    # Visualize inlier point matching
    inlier_img = cv2.drawMatches(
        img1, kp1, img2, kp2, inlier_matches, None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )

    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(inlier_img, cv2.COLOR_BGR2RGB))
    plt.title('Inlier Matches after RANSAC')
    plt.axis('off')
    plt.savefig(f"output/task4_inliers_{pair_name}.png", dpi=300, bbox_inches='tight')
    plt.show()

    # ---------- Step 5: Apply transformation ----------
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]

    corners2 = np.float32([[0, 0], [0, h2], [w2, h2], [w2, 0]]).reshape(-1, 1, 2)
    warped_corners = cv2.perspectiveTransform(corners2, H)

    all_corners = np.concatenate((
        np.float32([[0, 0], [0, h1], [w1, h1], [w1, 0]]).reshape(-1, 1, 2),
        warped_corners
    ), axis=0)

    [xmin, ymin] = np.int32(all_corners.min(axis=0).ravel())
    [xmax, ymax] = np.int32(all_corners.max(axis=0).ravel())
    shift = [-xmin, -ymin]
    output_size = (xmax - xmin, ymax - ymin)

    H_translation = np.array([
        [1, 0, shift[0]],
        [0, 1, shift[1]],
        [0, 0, 1]
    ])

    warped_img2 = cv2.warpPerspective(img2, H_translation @ H, output_size)

    img1_translated = np.zeros((output_size[1], output_size[0], 3), dtype=np.uint8)
    img1_translated[shift[1]:shift[1] + h1, shift[0]:shift[0] + w1] = img1

    # ---------- Step 6: Merge images ----------
    mask_warped = np.any(warped_img2 > 0, axis=2)
    mask_img1 = np.any(img1_translated > 0, axis=2)
    overlap = mask_warped & mask_img1

    result = img1_translated.copy()
    result[overlap] = (result[overlap] * 0.5 + warped_img2[overlap] * 0.5).astype(np.uint8)
    result[mask_warped & ~overlap] = warped_img2[mask_warped & ~overlap]

    # Visualize the final stitched image
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    plt.title('Stitched Image')
    plt.axis('off')
    plt.savefig(f"output/task4_stitched_{pair_name}.png", dpi=300, bbox_inches='tight')
    plt.show()

    return result


# -------------------- Run Task 4 --------------------
images = [
    ("castle1.jpg", "castle2.jpg", "castle"),
    ("monash1.jpg", "monash2.jpg", "monash"),
    ("singapore1.jpg", "singapore2.jpg", "singapore")
]

for img1_name, img2_name, pair_name in images:
    img1 = cv2.imread("images/" + img1_name)
    img2 = cv2.imread("images/" + img2_name)

    if img1 is None or img2 is None:
        print("Please ensure that the image file exists")
        print("Failed to read:", img1_name, img2_name)
        continue

    result = my_image_stitching(img1, img2, pair_name)

    if result is not None:
        cv2.imwrite(f"output/stitched_{pair_name}.jpg", result)
        print(f"{pair_name} stitching completed")
    else:
        print(f"{pair_name} stitching failed")

In [ ]:
import os
import shutil

src_folder = os.path.join(os.getcwd(), "output")

desktop = os.path.expanduser("~/Desktop")
dst_folder = os.path.join(desktop, "assignment_output")

os.makedirs(dst_folder, exist_ok=True)

for file in os.listdir(src_folder):
    src_file = os.path.join(src_folder, file)
    dst_file = os.path.join(dst_folder, file)
    shutil.copy(src_file, dst_file)

print(dst_folder)